In [ ]:
import ee
import geemap
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

    ee.Authenticate()
    ee.Initialize(project='')


feature_images = {
    "Beijing Shi": "project/assets/Beijing_Feature_Image",
    "Guangdong Sheng": "project/assets/Guangdong_Feature_Image",
    "Jiangsu Sheng": "project/assets/Jiangsu_Feature_Image",
    "Hebei Sheng": "project/assets/Hebei_Feature_Image",
    "Anhui Sheng": "project/assets/Anhui_Feature_Image",
    "Sichuan Sheng": "project/assets/Sichuan_Feature_Image",
    "Nei Mongol Zizhiqu": "project/assets/Inner_Mongolia_Feature_Image",
    "Shandong Sheng": "project/assets/Shandong_Feature_Image",
    "Hubei Sheng": "project/assets/Hubei_Feature_Image",
    "Guizhou Sheng": "project/assets/Guizhou_Feature_Image"
}

poi_csvs = {
    "Beijing Shi": r"D:\china_datacenter_clean - beijing.csv",
    "Guangdong Sheng": r"D:\china_datacenter_clean - guangdong.csv",
    "Jiangsu Sheng": r"D:\china_datacenter_clean - jiangsu.csv",
    "Hebei Sheng": r"D:\china_datacenter_clean - hebei.csv",
    "Anhui Sheng": r"D:\china_datacenter_clean - anhui.csv",
    "Sichuan Sheng": r"D:\china_datacenter_clean - sichuan.csv",
    "Nei Mongol Zizhiqu": r"D:\china_datacenter_clean - neimenggu.csv",
    "Shandong Sheng": r"D:\china_datacenter_clean - shandong.csv",
    "Hubei Sheng": r"D:\china_datacenter_clean - hubei.csv",
    "Guizhou Sheng": r"D:\china_datacenter_clean - guizhou.csv"
}

china_adm1 = ee.FeatureCollection("FAO/GAUL/2024/level1")

def calculate_metrics(cm_info):
    TN, FP, FN, TP = cm_info[0][0], cm_info[0][1], cm_info[1][0], cm_info[1][1]
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0.0
    return {
        "Accuracy": round(accuracy, 4),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1": round(f1, 4)
    }

def mosaic_province_images(province_list, image_dict):
    valid_prov_images = []
    for prov in province_list:
        try:
            prov_image = ee.Image(image_dict[prov])
            band_names = prov_image.bandNames().getInfo()
            if len(band_names) > 0:
                prov_boundary = china_adm1.filter(ee.Filter.eq('ADM1_NAME', prov)).geometry()
                prov_image_clip = prov_image.clip(prov_boundary)
                valid_prov_images.append(prov_image_clip)
        except:
            continue
    if len(valid_prov_images) == 0:
        raise Exception("No valid provincial images")
    elif len(valid_prov_images) == 1:
        return valid_prov_images[0]
    else:
        return ee.ImageCollection(valid_prov_images).mosaic()


print("="*60)
print("📌 LOPO Cross Validation (Nightlight-only)")
print("="*60)

lopo_results = []

for test_prov in feature_images.keys():

    print(f"\n==============================")
    print(f": {test_prov}")
    print(f"==============================")

  
    train_provs = [p for p in feature_images.keys() if p != test_prov]


    train_samples = ee.FeatureCollection([])
    for prov in train_provs:
        try:
            try:
                poi_fc = geemap.csv_to_ee(poi_csvs[prov], latitude='latitude', longitude='longitude')
            except:
                poi_fc = geemap.csv_to_ee(poi_csvs[prov], latitude='wgs84_lat', longitude='wgs84_lon')
            train_samples = train_samples.merge(poi_fc.map(lambda f: f.set('label',1)))
        except:
            continue

   
    train_boundary = ee.FeatureCollection([
        china_adm1.filter(ee.Filter.eq('ADM1_NAME', p)).first()
        for p in train_provs
    ]).geometry().dissolve()

    neg_train = ee.FeatureCollection.randomPoints(
        region=train_boundary,
        points=train_samples.size(),
        seed=42
    ).map(lambda f: f.set('label',0))
    train_samples = train_samples.merge(neg_train)
    

    train_image = mosaic_province_images(train_provs, feature_images).select(['Nightlight'])

  
    training_data = train_image.sampleRegions(
        collection=train_samples,
        properties=['label'],
        scale=10
    )
    classifier = ee.Classifier.smileRandomForest(numberOfTrees=100, seed=42).train(
        features=training_data,
        classProperty='label'
    )
  

    try:
        try:
            poi_fc_test = geemap.csv_to_ee(poi_csvs[test_prov], latitude='latitude', longitude='longitude')
        except:
            poi_fc_test = geemap.csv_to_ee(poi_csvs[test_prov], latitude='wgs84_lat', longitude='wgs84_lon')
        pos_test = poi_fc_test.map(lambda f: f.set('label',1))
    except:
        continue

    test_boundary = china_adm1.filter(ee.Filter.eq('ADM1_NAME', test_prov)).geometry()
    neg_test = ee.FeatureCollection.randomPoints(
        region=test_boundary,
        points=pos_test.size(),
        seed=99
    ).map(lambda f: f.set('label',0))
    test_samples = pos_test.merge(neg_test)
    print("Number of test samples:", test_samples.size().getInfo())

  
    test_image = ee.Image(feature_images[test_prov]).select(['Nightlight'])

 
    test_data = test_image.sampleRegions(
        collection=test_samples,
        properties=['label'],
        scale=10
    ).classify(classifier)

    cm = test_data.errorMatrix('label','classification')
    metrics = calculate_metrics(cm.getInfo())
    print(" Test results:", metrics)

    lopo_results.append({
        "Province": test_prov,
        "Accuracy": metrics['Accuracy'],
        "Precision": metrics['Precision'],
        "Recall": metrics['Recall'],
        "F1": metrics['F1']
    })


df_lopo = pd.DataFrame(lopo_results)
print("\n==============================")
print("📈 LOPO average results(Nightlight-only)")
print("==============================")
print(df_lopo.to_string(index=False))
print("\nAverage F1:", round(df_lopo['F1'].mean(),4))